# Player Archetype EDA

**Project Objective**:

Gather football player and team data to create a player scouting dashboard,
which utilises data analyses and Machine Learning to identify player architects,
and lesser-known players that may fit these.


**Plan**:
1. Data Prep
      - Import data and create a data dictionary
      - Check data quality (data types, missingness, duplicates)
2. EDA
      - Qualitative EDA (counts & distributions)
      - Quantitative EDA
        - Descriptive exploration (distribution, uniqueness)
        - Skew & Kurtosis
        - Outliers
        - Relationships & Redundant Columns (correlation, heirarchical clustering)
        - Check scaling
      - Combined EDA (check for relationships between Qualitative and Quantitative data)

See `project-plan.md` and `data-dictionary.md` in `/docs` for more detail.

## Setup & Imports

In [ ]:
import pandas as pd
from unidecode import unidecode
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import scipy.stats as stats
from statsmodels.graphics.mosaicplot import mosaic

from scripts.column_group_exploration import explore_column_group
from scripts.outlier_exploration import explore_outliers
from scripts.dimensionality_reduction import explore_dimensionality_reduction, add_reduced_dimensions
from scripts.constants import PER90_COUNT_COLUMNS

sns.set_theme(style="whitegrid")

# Constants
SEASON = '21-22'

## Pre-processing

In [ ]:
raw_player_data = pd.read_csv('../data/processed/2021-2022 Football Player Stats.csv', sep=';', encoding='latin1')
raw_player_data = raw_player_data.copy()
# raw_player_data['season'] = SEASON

invalid_character_count = raw_player_data.Player.str.contains('\\?', na=True, regex=True).sum()

if invalid_character_count > 0:
    print(f"WARNING: {invalid_character_count} `Player` values have a '?' in them.")

raw_player_data.head()

Some `Player` values have a question mark as a character. These are present in the underlying .csv; these will be left for the MVP, but an alternative data source for Phase2 may not have these.

There are very few metrics relevant to goalkeepers, and the requirements of the position are significantly different to outfield, making a single approach accross both groups inappropriate. Goalkeepers will therefore be excluded from this stage of the project. Goalkeeper-specific metrics may be available elsewhere, enabling a dedicated goalkeeper model & analysis to be produced at a later date.

Players with under 600 minutes at any given club will be excluded, to prevent skewing the data.

In [ ]:
# Rk is just an index column (sorted by last name). Names also include special characters, which will require normalisation
player_data = raw_player_data.drop('Rk', axis=1)
player_data['Player'] = player_data['Player'].apply(unidecode)

# Remove goalkeepers
player_data = player_data[~player_data.Pos.str.contains("GK")]

# Require a minimum of 600 minutes
player_data = player_data[player_data.Min >= 600]

duplicate_player_count = player_data.duplicated(subset="Player").sum()
duplicates_found = duplicate_player_count > 0

if duplicates_found:
    print(f"{duplicate_player_count} duplicate Player names found.")
    duplicate_players = player_data[player_data.duplicated(subset='Player', keep=False)].sort_values(by='Player')
    display(duplicate_players.head(10))
else:
    print("No duplicates found")

Players are duplicated when they have played for two clubs in a season.

**Options:**
1. Aggregate the metrics
2. Select only one entry (either the first or last club played-for, or the most-played for)
3. Treat both entries as separate
4. Some combination of these, accounting for league played in (e.g. aggregate if in the same league, else keep separate or select one)

**Decision:** _As we only have access to limited (one or two seasons worth of) data, we will aggregate the entries to avoid as much information loss as possible._

### Pre-Aggregation EDA on Duplicate Player Data

Before aggregating, we need to perform a quick sanity check. If a player appears twice, do their core attributes (e.g. `Nation`, `Age`, `Born`) also change?

`Nation` should rarely change (although it is possible that a player could change their declared nation).
`Age` could increment by 1, depending on how it is calculated (e.g. if it is calculated at the start of the season or at the moment of the row split).
`Born` should never change, and if so will indicate either a mistake or that they are two players with a common name.
It is unclear how `Pos` is currently calculated, or if it can change.

We will check for any variations in `Nation`, `Pos`, `Age`, and `Born` for players with multiple entries.

In [ ]:
if duplicates_found:
    unique_counts = duplicate_players.groupby('Player')[['Nation', 'Pos', 'Age', 'Born']].nunique()

    changes = (unique_counts > 1).sum().reset_index()
    changes.columns = ['Attribute', 'Players_With_Changes']
    changes['Players_With_No_Changes'] = unique_counts.shape[0] - changes['Players_With_Changes']

    melted = changes.melt(id_vars='Attribute', var_name='Status', value_name='Count')
    melted['Status'] = melted['Status'].replace({
        'Players_With_Changes': 'Changes across rows',
        'Players_With_No_Changes': 'Identical'
    })

    plt.figure(figsize=(10, 6))
    sns.set_theme(style='whitegrid')
    ax = sns.barplot(data=melted, x='Attribute', y='Count', hue='Status', palette='Set1')
    plt.title(f'Variability of Core Attributes among Duplicate Player Names (N={duplicate_player_count})', fontsize=14, fontweight='bold')
    plt.xlabel('Player Attribute', fontsize=12)
    plt.ylabel('Number of Unique Player Names', fontsize=12)
    plt.legend(title='Status')

    for p in ax.patches:
        if p.get_height() > 0:
            ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height() + 1), ha='center', va='bottom', fontsize=10, fontweight='bold')

    plt.tight_layout()
    plt.show()

3 Players have different Born values, suggesting these may actually be two different players with a common name.
The same is true for Age, although these are likely the same players.

In [ ]:
# find Players with multiple Born values
if duplicates_found:
    born_counts = duplicate_players.groupby('Player')['Born'].nunique()
    born_collision_players = born_counts[born_counts > 1].index

    born_collisions = duplicate_players[duplicate_players['Player'].isin(born_collision_players)].sort_values(by=['Player', 'Born'])

    display(born_collisions[['Player', 'Nation', 'Pos', 'Squad', 'Comp', 'Age', 'Born']])

In [ ]:
# find Players with different Ages, to check if they are the same
if duplicates_found:
    age_counts = duplicate_players.groupby('Player')['Age'].nunique()
    age_collision_players = age_counts[age_counts > 1].index

    age_collisions = duplicate_players[duplicate_players['Player'].isin(age_collision_players)].sort_values(by=['Player', 'Born'])

    if age_collisions.equals(born_collisions):
        print("The 'Born' overlaps are the same players as the 'Age' overlaps")

In [ ]:
# Check the nationality overlaps
if duplicates_found:
    nation_counts = duplicate_players.groupby('Player')['Nation'].nunique()
    nation_collision_players = nation_counts[nation_counts > 1].index

    nation_collisions = duplicate_players[duplicate_players['Player'].isin(nation_collision_players)].sort_values(by=['Player', 'Born'])

    display(nation_collisions[['Player', 'Nation', 'Pos', 'Squad', 'Comp', 'Age', 'Born']])

It is clear that the overlaps in Born, Age, and Nation are all due to different players sharing a common name. An identifying column can be made by combining these colums.

In [ ]:
# add to the player_data dataframe as processing, and to duplicate_players for further pre-aggregating EDA

player_data['player_identifier'] = (
        player_data['Player'] + '_' + player_data['Nation'] + '_' + player_data['Born'].astype(str))

duplicate_players['player_identifier'] = (
        duplicate_players['Player'] + '_' + duplicate_players['Nation'] + '_' + duplicate_players['Born'].astype(str))


player_data[['Player', 'Nation', 'Age', 'Born', 'player_identifier']]

In [ ]:
# Rerunning pre-aggregation checks using this new identifier:
if duplicates_found:
    unique_counts = duplicate_players.groupby('player_identifier')[['Nation', 'Pos', 'Age', 'Born']].nunique()

    changes = (unique_counts > 1).sum().reset_index()
    changes.columns = ['Attribute', 'Players_With_Changes']
    changes['Players_With_No_Changes'] = unique_counts.shape[0] - changes['Players_With_Changes']

    melted = changes.melt(id_vars='Attribute', var_name='Status', value_name='Count')
    melted['Status'] = melted['Status'].replace({
        'Players_With_Changes': 'Changes across rows',
        'Players_With_No_Changes': 'Identical'
    })

    plt.figure(figsize=(10, 6))
    ax = sns.barplot(data=melted, x='Attribute', y='Count', hue='Status', palette='Set1')
    plt.title(f'Variability of Core Attributes among Duplicate Player Identifiers (N={duplicate_player_count})', fontsize=14, fontweight='bold')
    plt.xlabel('Player Attribute', fontsize=12)
    plt.ylabel('Number of Unique Player Names', fontsize=12)
    plt.legend(title='Status')

    for p in ax.patches:
        if p.get_height() > 0:
            ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height() + 1), ha='center', va='bottom', fontsize=10, fontweight='bold')

    plt.tight_layout()
    plt.show()

These has removed the majority of the crossovers, however there are still a  small number (4) of duplicate players with multiple positions.

In [ ]:
# find Players with multiple Pos values
if duplicates_found:
    pos_counts = duplicate_players.groupby('player_identifier')['Pos'].nunique()
    pos_collision_players = pos_counts[pos_counts > 1].index

    pos_collisions = duplicate_players[duplicate_players['player_identifier'].isin(pos_collision_players)].sort_values(by=['player_identifier', 'Born'])

    display(pos_collisions[['Player', 'Nation', 'Pos', 'Squad', 'Comp', 'Age', 'Born', 'player_identifier']])

This is due to players playing a different set of positions at different clubs. This leaves 5 options:
1. Treat the entries as distinct, as different positions are likely to have different metric profiles (will lead to multiple rows for some players, overweighting them in modelling)
2. Track all the positions played across the season, however this could lead to skewed data (as different positions will naturally lead to different performance metrics, but will be collapsed)
3. Remove the players entirely from the dataset (may skew the data as high-performers may be more likely to transfer)
4. Acquire more granular position data (ideal, but outside the scope of the MVP for this project)

For the MVP, and as the player's `Pos` values are already aggregated across the dataset (i.e. players can already have multiple positions), tracking all the positions played is justified. However, acquiring more granular detail will be an important discovery step for Phase 2.

---

The final major consideration for aggregating multi-row players is how to handle multiple `Comp` and `Club` values. While these won't be included as part of the player architecture modelling, they are very likely to be relevant to player value prediction. One-hot-encoding will be required for modelling anyway, but Comp can instead be transformed to a `{Comp Name}_pct_mins`, taking the percentage of the player's minute played in each of the 5 leagues included in the dataset. In order to scale this to more leagues for future modelling, during Phase2 `Comp` can instead be replaced by some measure of league-level coefficient (e.g. UEFA coefficient for European clubs), with a weighted average being taken according to the `pct_mins` a player played in each league.

`Club` cannot be simply one-hot-encoded, as cardinality will be too high. For the MVP, club will be excluded as a modelling feature and array aggregated for post-modelling analyis only. For Phase2 club-level metrics (e.g. ELO, league finish, etc) can be used instead.


### Aggregation

**Decisions:**
| Consideration | MVP Solution | Phase2 Improvement |
| ------------- | ------------ | ------------------ |
| `Pos` | Combine positions played across the season, and weighted-average (by minutes) the stats. One-hot-encode for modelling. | Acquire granular position level data (e.g. minutes player per position, or stats split by position player) |
| `Comp` | Create a column for each league, showing the percentage of minutes played in each. Exclude from clustering. | Replace Comp name from feature set with league-level data, such as coefficients. |
| `Club` | Array aggregate (as with `Pos`), but exclude from all modelling. | As with `Comp`, replace Club names with club-level metrics (ELO, league finish, etc) |

---

As per the data-dictionary.md, the other columns are counts (which can be summed to aggregate), or ratios and percentages. The latter two will need re-calculating to ensure a weighted-average. The majority of these can be recalculated from existing (summed) columns, however three cannot:
- Min% (Playing Time): Requires total team minutes which is not present in the dataset.
- On-Off (Playing Time): Requires team-level off-pitch performance stats.
- Dist (Shooting): Recalculating a true average shot distance requires individual shot distances or total shot counts/distances which are not available.

The aggregation strategy for these will be:
- Min%: Drop the column
- On-Off: Drop the column
- Dist: take a weighted average based on number of shots. e.g. (dist_at_clubA * shots_at_clubA + dist_at_clubB * shots_at_clubB)/(shots_at_clubA + shots_at_clubB)

In [ ]:
# Aggregate to one row per player based on the strategy above:

player_data_clean = player_data.copy()

# estimate a weighted shot distance column
player_data_clean['ShoDist_weighted'] = player_data_clean['ShoDist'] * player_data_clean['Shots']

# split minutes played into one for each league
comps = player_data_clean['Comp'].dropna().unique()
comp_min_cols = []
for comp in comps:
    min_col = f"{comp}_pct_mins"
    player_data_clean[min_col] = player_data_clean['Min'].where(player_data_clean['Comp'] == comp, 0)
    comp_min_cols.append(min_col)

# define the aggregation strategy in a dictionary
agg_dict = {}

numeric_cols = player_data_clean.select_dtypes(include='number').columns
recalc_cols = ['90s', 'SoT%', 'G/Sh', 'G/SoT', 'PasTotCmp%', 'PasShoCmp%', 'PasMedCmp%', 'PasLonCmp%', 'TklDri%', 'Press%', 'DriSucc%', 'Rec%', 'AerWon%', 'ShoDist']
exclude_from_sum = recalc_cols + ['Age', 'Born']

agg_dict['Player'] = 'first'
agg_dict['Nation'] = 'first'
agg_dict['Born'] = 'first'
agg_dict['Age'] = 'max'
agg_dict['Pos'] = lambda x: ''.join(sorted(x.unique()))
agg_dict['Squad'] = lambda x: ', '.join(sorted(x.unique()))
agg_dict['Comp'] = lambda x: ', '.join(sorted(x.unique()))

for col in numeric_cols:
    if col not in exclude_from_sum:
        agg_dict[col] = 'sum'

# aggregate using the dictionary
aggregated_player_data = player_data_clean.groupby('player_identifier').agg(agg_dict)

# determine the primary position (first position listed in the row with maximum minutes)
idx_max_min = player_data_clean.groupby('player_identifier')['Min'].idxmax()
primary_pos_map = player_data_clean.loc[idx_max_min].set_index('player_identifier')['Pos'].str[:2]
aggregated_player_data['Primary_Pos'] = aggregated_player_data.index.map(primary_pos_map)

# convert the league minutes column to a percentage of minutes played
for pct_col in comp_min_cols:
    aggregated_player_data[pct_col] = (aggregated_player_data[pct_col] / aggregated_player_data['Min']).fillna(0)

# Recalculate ratios and percentages
aggregated_player_data['90s'] = aggregated_player_data['Min'] / 90
aggregated_player_data['SoT%'] = (aggregated_player_data['SoT'] / aggregated_player_data['Shots'] * 100).fillna(0)
aggregated_player_data['G/Sh'] = (aggregated_player_data['Goals'] / aggregated_player_data['Shots']).fillna(0)
aggregated_player_data['G/SoT'] = (aggregated_player_data['Goals'] / aggregated_player_data['SoT']).fillna(0)
aggregated_player_data['PasTotCmp%'] = (aggregated_player_data['PasTotCmp'] / aggregated_player_data['PasTotAtt'] * 100).fillna(0)
aggregated_player_data['PasShoCmp%'] = (aggregated_player_data['PasShoCmp'] / aggregated_player_data['PasShoAtt'] * 100).fillna(0)
aggregated_player_data['PasMedCmp%'] = (aggregated_player_data['PasMedCmp'] / aggregated_player_data['PasMedAtt'] * 100).fillna(0)
aggregated_player_data['PasLonCmp%'] = (aggregated_player_data['PasLonCmp'] / aggregated_player_data['PasLonAtt'] * 100).fillna(0)
aggregated_player_data['TklDri%'] = (aggregated_player_data['TklDri'] / aggregated_player_data['TklDriAtt'] * 100).fillna(0)
aggregated_player_data['Press%'] = (aggregated_player_data['PresSucc'] / aggregated_player_data['Press'] * 100).fillna(0)
aggregated_player_data['DriSucc%'] = (aggregated_player_data['DriSucc'] / aggregated_player_data['DriAtt'] * 100).fillna(0)
aggregated_player_data['Rec%'] = (aggregated_player_data['Rec'] / aggregated_player_data['RecTarg'] * 100).fillna(0)
aggregated_player_data['AerWon%'] = (aggregated_player_data['AerWon'] / (aggregated_player_data['AerWon'] + aggregated_player_data['AerLost']) * 100).fillna(0)
aggregated_player_data['ShoDist'] = (aggregated_player_data['ShoDist_weighted'] / aggregated_player_data['Shots']).fillna(0)

# create the estimate for weighted average shot distance
aggregated_player_data = aggregated_player_data.drop(columns=['ShoDist_weighted'])
aggregated_player_data = aggregated_player_data.replace([np.inf, -np.inf], 0)

aggregated_player_data = aggregated_player_data.reset_index().copy()

print(f"Original shape: {player_data.shape}")
print(f"Aggregated shape: {aggregated_player_data.shape}")

In [ ]:
if duplicates_found:
    display(aggregated_player_data[aggregated_player_data.player_identifier.isin(duplicate_players.player_identifier)].head(10))
else:
    display(aggregated_player_data.head(10))

In [ ]:
remaining_dupes = aggregated_player_data[aggregated_player_data.player_identifier.isin(duplicate_players.player_identifier)]
duplicate_player_count = remaining_dupes.duplicated(subset="player_identifier").sum()

print(f"{duplicate_player_count} duplicate players remaining")

## Exploratory Analysis

In [ ]:
qualitative_columns = list(aggregated_player_data.select_dtypes(include='string').columns)
quantitative_columns = list(aggregated_player_data.select_dtypes(include='number').columns)

### Qualitative Columns (Pos, Nation, Comp, and Squad)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Target top 5 nations
target_nations = {"ESP", "FRA", "GER", "ITA", "ENG"}

# ---------------------------------------------------------
# 1. Top 15 Nations by Player Count (Conditional Blue/Red)
# ---------------------------------------------------------
top_nations = aggregated_player_data["Nation"].value_counts().head(15)
nation_colors = {
    nation: "#008fd5" if nation in target_nations else "slategrey"
    for nation in top_nations.index
}

sns.barplot(
    x=top_nations.values,
    y=top_nations.index,
    hue=top_nations.index,
    palette=nation_colors,
    ax=axes[0, 0],
)
axes[0, 0].set_title(
    "Top 15 Nations by Player Count", fontsize=12, fontweight="bold"
)
axes[0, 0].set_xlabel("Number of Players")

# ---------------------------------------------------------
# 2. Player Count by Competition (Custom Palette with Fallback)
# ---------------------------------------------------------
# Define custom colors for competitions (add your specific values here)
comp_palette_map = {
    "Premier League": "darkviolet",
    "La Liga": "gold",
    "Bundesliga": "red",
    "Serie A": "limegreen",
    "Ligue 1": "royalblue",
}

comp_counts = aggregated_player_data["Comp"].value_counts()
comp_colors = {
    comp: comp_palette_map.get(comp, "slategrey") for comp in comp_counts.index
}

sns.barplot(
    x=comp_counts.values,
    y=comp_counts.index,
    hue=comp_counts.index,
    palette=comp_colors,
    ax=axes[0, 1],
)
axes[0, 1].set_title(
    "Player Count by Competition", fontsize=12, fontweight="bold"
)
axes[0, 1].set_xlabel("Number of Players")

# ---------------------------------------------------------
# 3. Top 10 Positions (Custom Palette with Fallback)
# ---------------------------------------------------------
# Define custom colors for positions (add your specific values here)
pos_palette_map = {
    "DF": "gold",
    "MF": "limegreen",
    "FW": "dodgerblue",
    "GK": "darkorange",

}

top_pos = aggregated_player_data["Primary_Pos"].value_counts().head(10)
pos_colors = {
    pos: pos_palette_map.get(pos, "slategrey") for pos in top_pos.index
}

sns.barplot(
    x=top_pos.values,
    y=top_pos.index,
    hue=top_pos.index,
    palette=pos_colors,
    ax=axes[1, 0],
)
axes[1, 0].set_title(
    "Primary Positions by Player Count", fontsize=12, fontweight="bold"
)
axes[1, 0].set_xlabel("Number of Players")

# ---------------------------------------------------------
# 4. Top 15 Squads Colored by Competition
# ---------------------------------------------------------
top_squad_names = (
    aggregated_player_data["Squad"].value_counts().head(15).index
)

# Filter dataset to top 15 squads and aggregate count per Squad and Comp
top_squads_df = (
    aggregated_player_data[
        aggregated_player_data["Squad"].isin(top_squad_names)
    ]
    .groupby(["Squad", "Comp"])
    .size()
    .reset_index(name="Count")
    .sort_values(by="Count", ascending=False)
)

sns.barplot(
    data=top_squads_df,
    x="Count",
    y="Squad",
    hue="Comp",
    palette=comp_colors,  # Reuses competition color map from Chart 2
    dodge=False,
    ax=axes[1, 1],
)
axes[1, 1].set_title(
    "Top 15 Squads by Player Count (by Comp)", fontsize=12, fontweight="bold"
)
axes[1, 1].set_xlabel("Number of Players")

plt.tight_layout()
plt.show()

In [ ]:
categories = {
    "Nation": "Top 5 Nations",
    "Comp": "Top 5 Competitions",
    "Primary_Pos": "Top 5 Positions",
    "Squad": "Top 5 Squads",
}

for col, title in categories.items():
    counts = aggregated_player_data[col].value_counts().head(5)
    total_valid = aggregated_player_data[col].count()

    summary_df = pd.DataFrame(
        {
            col: counts.index,
            "Count": counts.values,
            "Percentage (%)": ((counts.values / total_valid) * 100).round(2),
        }
    )

    print(f"\n{'='*15} {title} {'='*15}")
    print(summary_df.to_string(index=False))

In [ ]:
def clean_positions(pos_str):
  if not isinstance(pos_str, str) or pd.isna(pos_str):
      return ""
# Keep only alphabetical characters
  clean_str = "".join([c for c in pos_str if c.isalpha()])
# Chunk into 2-letter position codes (e.g., 'MFDFDF' -> ['MF', 'DF', 'DF'])
  chunks = [clean_str[i:i+2] for i in range(0, len(clean_str), 2) if len(clean_str[i:i+2]) == 2]
# Deduplicate and sort alphabetically
  unique_sorted = sorted(list(set(chunks)))
  return ", ".join(unique_sorted)

# Apply position cleanup to the dataset
aggregated_player_data['Pos'] = aggregated_player_data['Pos'].apply(clean_positions)

# Count the number of different positions played by each player (e.g., 'DF, MF' -> 2)
aggregated_player_data['Num_Pos'] = aggregated_player_data['Pos'].apply(lambda x: len(x.split(', ')) if x else 0)

multi_pos_players = aggregated_player_data[aggregated_player_data.Num_Pos > 1]

display(multi_pos_players.sample(5))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot total counts for each number of positions played
sns.countplot(
    data=aggregated_player_data,
    x="Num_Pos",
    hue="Num_Pos",
    palette="crest",
    ax=ax1,
)

# Remove the redundant legend generated by hue
if ax1.get_legend():
    ax1.get_legend().remove()

# Calculate total non-zero/valid entries for percentages
total_players = len(aggregated_player_data)

# Add count and percentage annotations above each bar
for p in ax1.patches:
    height = p.get_height()
    if height > 0:
        pct = (height / total_players) * 100
        ax1.annotate(
            f"{int(height):,}\n({pct:.1f}%)",
            (p.get_x() + p.get_width() / 2.0, height),
            ha="center",
            va="bottom",
            xytext=(0, 4),
            textcoords="offset points",
            fontsize=10,
            fontweight="bold",
        )

ax1.set_title(
    "Distribution of Number of Positions Played per Player",
    fontsize=13,
    fontweight="bold",
    pad=15,
)
ax1.set_xlabel("Number of Positions Played", fontsize=11, fontweight="bold")
ax1.set_ylabel("Number of Players", fontsize=11, fontweight="bold")

# Expand top y-limit to prevent annotation cutoff
ylim1 = ax1.get_ylim()
ax1.set_ylim(ylim1[0], ylim1[1] * 1.15)

# Plot total counts for each value in 'Pos' column (sorted by count)
pos_order = multi_pos_players['Pos'].value_counts().index
sns.countplot(
    data=aggregated_player_data,
    x="Pos",
    hue="Pos",
    order=pos_order,
    palette="crest",
    ax=ax2,
)

# Remove the redundant legend generated by hue
if ax2.get_legend():
    ax2.get_legend().remove()

# Add count and percentage annotations above each bar for 'Pos' count plot
for p in ax2.patches:
    height = p.get_height()
    if height > 0:
        pct = (height / total_players) * 100
        ax2.annotate(
            f"{int(height):,}\n({pct:.1f}%)",
            (p.get_x() + p.get_width() / 2.0, height),
            ha="center",
            va="bottom",
            xytext=(0, 4),
            textcoords="offset points",
            fontsize=10,
            fontweight="bold",
        )

ax2.set_title(
    "Distribution of Players across Positions",
    fontsize=13,
    fontweight="bold",
    pad=15,
)
ax2.set_xlabel("Positions Played", fontsize=11, fontweight="bold")
ax2.set_ylabel("Number of Players", fontsize=11, fontweight="bold")
plt.setp(ax2.get_xticklabels(), rotation=30, ha="right")

# Expand top y-limit to prevent annotation cutoff
ylim2 = ax2.get_ylim()
ax2.set_ylim(ylim2[0], ylim2[1] * 1.15)

plt.tight_layout()
plt.show()

**Findings:**
There are far more Spanish and French players in Europe's top 5 leagues than other nationalities. Some way behind these are Germany, Italy, and England, before another significant drop off to other nationalities. Brazil, Argentina and Senegal are the only non-European nations in the Top 15.

La Liga and Serie A, 1 have the most players respresented, with Ligue 1 and the Premier League a little further behind. Bundesliga unsurprisingly has the least (due to only having 18 clubs in the league). Only a very small proportion of players have played in multiple leagues in the season, with the Premier League to Seria A being the most common switch.

42% of players are defenders, with 34% midfielders, and 24% forwards. 25.3% of players have also played a secondary position, with the vast majority (19.1% of all players) of these being players playing FW and MF

Italian, French, and Spanish clubs dominate the top squads by player count, with just two Premier League (Leicester and Watford), and no Bundesliga clubs. Paris Saint-Germain is the only elite-level club that stands out in the top 15.

In [ ]:
# Perform a Chi-squared test to check for independence between the nations represented and each league
# Excluding the 5 Nations that the league's themselves are in

df_exploded = aggregated_player_data[
    ['player_identifier', 'Nation', 'Comp']
].copy()
df_exploded = df_exploded[
    ~df_exploded.Nation.isin(['ENG', 'GER', 'FRA', 'ESP', 'ITA'])
]

df_exploded['Nation'] = df_exploded['Nation'].str.split(', ')
df_exploded = df_exploded.explode('Nation')
df_exploded['Comp'] = df_exploded['Comp'].str.split(', ')
df_exploded = df_exploded.explode('Comp')

# Get Top 15 nations
top_15_nations = df_exploded['Nation'].value_counts().head(15).index
df_filtered = df_exploded[df_exploded['Nation'].isin(top_15_nations)].copy()
df_filtered = df_filtered.reset_index(drop=True)

# Sort for consistent ordering in plots
df_filtered = df_filtered.sort_values(by=['Nation', 'Comp'])

# 2. Chi-Squared Test of Independence
contingency_table = pd.crosstab(df_filtered['Nation'], df_filtered['Comp'])
chi2, p_val, dof, expected = stats.chi2_contingency(contingency_table)

# Calculate Pearson Residuals
residuals = (contingency_table - expected) / np.sqrt(expected)

# --- NEW: Build Annotation Matrix for Heatmap (Residual + Count) ---
annot_matrix = pd.DataFrame(
    '', index=residuals.index, columns=residuals.columns
)
for r in residuals.index:
  for c in residuals.columns:
    count = contingency_table.loc[r, c]
    res_val = residuals.loc[r, c]
    annot_matrix.loc[r, c] = f'{res_val:.2f}\n({count})'

print('--- Chi-Squared Test Results ---')
print(f'Chi-squared Statistic: {chi2:.4f}')
print(f'p-value: {p_val:.4e}')
print(f'Degrees of Freedom: {dof}')
print('\nSignificance:')
if p_val < 0.05:
  print(
      'Statistically Significant (p < 0.05). Nations are NOT distributed'
      ' proportionally across leagues.'
  )
else:
  print(
      'Not Statistically Significant (p >= 0.05). Nations are distributed'
      ' proportionally across leagues.'
  )

In [ ]:
fig = plt.figure(figsize=(24, 10))
gs = fig.add_gridspec(1, 2, width_ratios=[1.2, 1])

# Plot A: Residual Heatmap with Counts
ax_heatmap = fig.add_subplot(gs[0, 0])
sns.heatmap(
    residuals,
    annot=annot_matrix,  # Custom string matrix with count in brackets
    fmt='',  # Disable default numeric formatting
    cmap='RdYlGn',
    center=0,
    cbar_kws={'label': 'Pearson Residuals (Significance threshold: ±2.0)'},
    ax=ax_heatmap,
)
ax_heatmap.set_title(
    'Pearson Residuals (Over/Under Representation Heatmap)\nResiduals with'
    ' Raw Counts (in Brackets)',
    fontsize=14,
    fontweight='bold',
)
ax_heatmap.set_xlabel('Competition / League')
ax_heatmap.set_ylabel('Nation')

# Plot B: Mosaic Plot (Flipped: Comp on x-axis, Nation on y-axis)
ax_mosaic = fig.add_subplot(gs[0, 1])


def cell_properties(key):
  # key is now a tuple: (Comp, Nation)
  comp, nation = key
  if nation in residuals.index and comp in residuals.columns:
    res = residuals.loc[nation, comp]
    if res > 2.0:
      color = (
          '#2ca02c'  # Statistically significant over-representation (Dark Green)
      )
    elif res < -2.0:
      color = (
          '#d62728'  # Statistically significant under-representation (Dark Red)
      )
    elif res > 0:
      color = (
          '#a1d99b'  # Mild/non-significant over-representation (Light Green)
      )
    else:
      color = '#ff9896'  # Mild/non-significant under-representation (Light Red)
  else:
    color = '#f0f0f0'
  return {'color': color, 'edgecolor': 'white'}


def cell_labelizer(key):
  # key is (Comp, Nation) -> return only the nation name
  comp, nation = key
  if nation in contingency_table.index and comp in contingency_table.columns:
    # Only label non-zero cells to avoid clutter
    if contingency_table.loc[nation, comp] > 0:
      return nation
  return ''


# Create Mosaic Plot using statsmodels with ['Comp', 'Nation'] order
mosaic(
    df_filtered,
    ['Comp', 'Nation'],
    properties=cell_properties,
    labelizer=cell_labelizer,  # Custom labelizer showing only nation names
    ax=ax_mosaic,
)

# Rotate labels on mosaic plot for legibility
plt.setp(ax_mosaic.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.show()

**Findings:**

Even excluding the nations that are home to the top 5 leagues, the nations represented are not independent of the league.
- In the Bundesliga, Austrian and Swiss Players in particular are over-represented vs the other Top 5 leagues, while South Americans (from Argentina, Brazil, and Uruguay) are the most under-represented.
- Conversely, Argentineans and Uruguayans are significantly over-represented in La Liga
- Senegalese and Ivorians are the over-represented in Ligue 1
- The Premier League and Serie A has fewer extremes. Portuguese, Danish, and Brazilian players are over-represented in the former, with Argentineans, Danes and Croats under-represented in the former. In Serie A, Polish and Croatian players are over-represented.

### Qualitative EDA

The majority of metrics are raw counts, which can be skewed simply by playing time (e.g., a player who plays twice as many minutes may have more total passes than a player who has had limited playtime but makes far more passes per game). These counts can therefore be normalised by calculating the 'per 90 min' equivalents. Ratios, Percentages, and already-calculated-averages (e.g goals-per-shot) do not require further normalisation.

#### Age

`Age` and `Born` show essentially the same data, so the latter will be dropped.

In [ ]:
# Age EDA

# Drop 'Born' if it hasn't been dropped already
if 'Born' in aggregated_player_data.columns:
    aggregated_player_data.drop('Born', axis=1, inplace=True)

# 1. Histogram with Rug Plot and Boxplot of the Age column
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histogram with Rug Plot
sns.histplot(data=aggregated_player_data, x='Age', kde=True, ax=axes[0], color='skyblue', edgecolor='black')
sns.rugplot(data=aggregated_player_data, x='Age', ax=axes[0], color='red', alpha=0.6, height=0.05)
axes[0].set_title('Distribution of Player Age (Histogram & Rug Plot)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Age', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)

# Boxplot
sns.boxplot(data=aggregated_player_data, y='Age', ax=axes[1], color='lightgreen')
axes[1].set_title('Boxplot of Player Age', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Age', fontsize=12)

plt.tight_layout()
plt.show()

# 2. Ridgeline Plots for Age by Primary_Pos and Comp
def draw_ridgeline_plot(df, group_col, value_col, order=None, title=None, palette='viridis'):
    # Filter out missing values in group or value columns
    plot_df = df[[group_col, value_col]].dropna().copy()
    
    # If order is not provided, sort groups by their mean value
    if order is None:
        order = plot_df.groupby(group_col)[value_col].mean().sort_values().index.tolist()
    else:
        # Keep only groups that actually exist in the data
        order = [g for g in order if g in plot_df[group_col].unique()]
        
    plot_df = plot_df[plot_df[group_col].isin(order)]
    plot_df[group_col] = pd.Categorical(plot_df[group_col], categories=order, ordered=True)
    
    # Store current RC parameters to restore later
    old_rc = plt.rcParams.copy()
    
    # Set style for ridgeline (overlapping, transparent background)
    sns.set_theme(style='white', rc={'axes.facecolor': (0, 0, 0, 0)})
    
    # Create the FacetGrid
    g = sns.FacetGrid(
        plot_df, 
        row=group_col, 
        hue=group_col, 
        aspect=10, 
        height=1.2, 
        palette=palette,
        row_order=order,
        hue_order=order
    )
    
    # Map the KDE plots (densities)
    g.map(sns.kdeplot, value_col, fill=True, alpha=0.75, lw=1.5, clip_on=False)
    g.map(sns.kdeplot, value_col, color='white', lw=2, clip_on=False)
    
    # Map the reference line
    g.refline(y=0, linewidth=1, linestyle='-', color='gray', clip_on=False)
    
    # Add custom labels (showing category name and its mean value)
    def label_axes(x, color, label):
        ax = plt.gca()
        mean_val = x.mean()
        ax.text(
            0, 0.2, f'{label} (Mean: {mean_val:.1f})', 
            fontweight='bold', 
            color=color,
            ha='left', 
            va='center', 
            transform=ax.transAxes,
            fontsize=12
        )
        
    g.map(label_axes, value_col)
    
    # Set the subplots to overlap slightly
    g.figure.subplots_adjust(hspace=-0.4)
    
    # Remove axis elements that interfere with ridgeline overlap
    g.set_titles('')
    g.set(yticks=[], ylabel='')
    g.despine(bottom=True, left=True)
    
    # Set common x-axis label
    g.set_xlabels(value_col, fontsize=12, fontweight='bold')
    
    if title:
        g.figure.suptitle(title, fontsize=16, fontweight='bold', y=0.98)
        
    plt.show()
    
    # Restore standard theme and RC parameters
    plt.rcParams.update(old_rc)
    sns.set_theme(style='whitegrid')

# Create ridgeline plot for Primary_Pos
draw_ridgeline_plot(
    df=aggregated_player_data, 
    group_col='Primary_Pos', 
    value_col='Age', 
    order=['GK', 'DF', 'MF', 'FW'], 
    title='Age Distribution by Primary Position', 
    palette='crest'
)

# For Comp, let's handle players playing in single leagues.
# Some players might have 'Ligue 1, Premier League' etc. if they transferred.
# We will focus on single primary competition values for cleaner ridgeline plots.
single_comps = ['Premier League', 'La Liga', 'Bundesliga', 'Serie A', 'Ligue 1']
df_single_comp = aggregated_player_data[aggregated_player_data['Comp'].isin(single_comps)]

# Create ridgeline plot for Comp
draw_ridgeline_plot(
    df=df_single_comp, 
    group_col='Comp', 
    value_col='Age', 
    title='Age Distribution by Competition (Top 5 Leagues)', 
    palette='flare'
)

# 3. Statistical Tests for Age Distribution across Categories
print('=== Statistical Tests for Age Distribution ===')

# --- A. Test for Primary_Pos ---
# Drop any NaN ages and group by Primary_Pos
df_pos_test = aggregated_player_data[['Primary_Pos', 'Age']].dropna()
# Get lists of ages for each primary position
pos_groups = [group['Age'].values for name, group in df_pos_test.groupby('Primary_Pos')]

if len(pos_groups) > 1:
    # Parametric ANOVA
    f_stat, p_anova_pos = stats.f_oneway(*pos_groups)
    # Non-parametric Kruskal-Wallis
    h_stat, p_kruskal_pos = stats.kruskal(*pos_groups)
    
    print('\n1. Age Distribution by Primary Position:')
    print(f'   One-way ANOVA: F-statistic = {f_stat:.4f}, p-value = {p_anova_pos:.4e}')
    print(f'   Kruskal-Wallis: H-statistic = {h_stat:.4f}, p-value = {p_kruskal_pos:.4e}')
    
    if p_kruskal_pos < 0.05:
        print('   -> Statistically Significant (p < 0.05). Age distribution differs significantly between positions.')
    else:
        print('   -> Not Statistically Significant (p >= 0.05). No significant age difference between positions.')
else:
    print('\n1. Age Distribution by Primary Position: Not enough groups to perform statistical tests.')

# --- B. Test for Comp (Top 5 Leagues) ---
# Filter for single primary competition values
df_comp_test = df_single_comp[['Comp', 'Age']].dropna()
comp_groups = [group['Age'].values for name, group in df_comp_test.groupby('Comp')]

if len(comp_groups) > 1:
    # Parametric ANOVA
    f_stat_comp, p_anova_comp = stats.f_oneway(*comp_groups)
    # Non-parametric Kruskal-Wallis
    h_stat_comp, p_kruskal_comp = stats.kruskal(*comp_groups)
    
    print('\n2. Age Distribution by Competition (Top 5 Leagues):')
    print(f'   One-way ANOVA: F-statistic = {f_stat_comp:.4f}, p-value = {p_anova_comp:.4e}')
    print(f'   Kruskal-Wallis: H-statistic = {h_stat_comp:.4f}, p-value = {p_kruskal_comp:.4e}')
    
    if p_kruskal_comp < 0.05:
        print('   -> Statistically Significant (p < 0.05). Age distribution differs significantly between leagues.')
    else:
        print('   -> Not Statistically Significant (p >= 0.05). No significant age difference between leagues.')
else:
    print('\n2. Age Distribution by Competition: Not enough groups to perform statistical tests.')

In [ ]:
# Plot the distribution of minutes played by Age

aggregated_player_data['Age'] = aggregated_player_data['Age'].astype('Int64')

# Calculate the mean minutes for each age (sorted)
age_means = (
   aggregated_player_data.groupby('Age')['Min']
   .mean()
   .sort_index()
)

# Calculate the sum of minutes for each age (sorted) for the bar plot
age_minutes_sum = (
   aggregated_player_data.groupby('Age')['Min']
   .sum()
   .reset_index()
)
age_minutes_sum['Age'] = age_minutes_sum['Age'].astype(int)

# 2. Create side-by-side subplots
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Plot 1: Strip Plot with Average Line Overlay (on axes[0]) ---
# Strip Plot (reduced alpha to 0.5 for better visibility)
sns.stripplot(
   data=aggregated_player_data,
   x='Age',
   y='Min',
   hue='Primary_Pos',
   jitter=0.2,
   alpha=0.8,
   ax=axes[0]
)

# Overlay Line Plot of the Mean
# We use range(len(age_means)) to match the 0, 1, 2... categorical indices of the strip plot
axes[0].plot(
   range(len(age_means)),
   age_means.values,
   color='red',
   linewidth=2.5,
    alpha = 0.5,
   label='Average Minutes'
)

axes[0].set_title('Minutes Played by Age and Position (with Average)', fontsize=12, fontweight='bold', pad=10)
# Update the legend to include the average line
axes[0].legend(title='Position / Trend', bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0.)
axes[0].tick_params(axis='x', rotation=45)

# --- Plot 2: Bar Plot of Sum of Minutes (on axes[1]) ---
sns.barplot(
   data=age_minutes_sum,
   x='Age',
   y='Min',
   color='royalblue',
   edgecolor='black',
   alpha=0.8,
   ax=axes[1]
)
axes[1].set_title('Total Minutes Played by Age', fontsize=12, fontweight='bold', pad=10)
axes[1].set_ylabel('Total Minutes')
axes[1].get_yaxis().set_major_formatter(plt.FuncFormatter(lambda x, loc: "{:,}".format(int(x))))
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Ensure clean data
df_clean = aggregated_player_data.dropna(subset=['Age', 'Min', 'Comp', 'Primary_Pos']).copy()
df_clean['Age'] = df_clean['Age'].astype(int)
df_clean['Min'] = df_clean['Min'].astype(float)

# =====================================================================
# PART 1: Plotting Total Minutes Played (Min) over Age by Category
# =====================================================================

fig, axes = plt.subplots(2, 1, figsize=(16, 12))

# 1. Total Minutes by Age & Competition (Top 5 Leagues)
single_comps = ['Premier League', 'La Liga', 'Bundesliga', 'Serie A', 'Ligue 1']
df_single_comp = df_clean[df_clean['Comp'].isin(single_comps)]

minutes_by_age_comp = (
   df_single_comp.groupby(['Age', 'Comp'])['Min']
   .sum()
   .reset_index()
)

sns.lineplot(
   data=minutes_by_age_comp,
   x='Age',
   y='Min',
   hue='Comp',
   marker='o',
   linewidth=2.5,
   ax=axes[0]
)
axes[0].set_title('Total Minutes Played by Age and Competition (Top 5 Leagues)', fontsize=14, fontweight='bold', pad=15)
axes[0].set_ylabel('Total Minutes Played', fontsize=12)
axes[0].get_yaxis().set_major_formatter(plt.FuncFormatter(lambda x, loc: "{:,}".format(int(x))))
axes[0].grid(True, linestyle='--', alpha=0.5)

# 2. Total Minutes by Age & Primary Position
minutes_by_age_pos = (
   df_clean.groupby(['Age', 'Primary_Pos'])['Min']
   .sum()
   .reset_index()
)

sns.lineplot(
   data=minutes_by_age_pos,
   x='Age',
   y='Min',
   hue='Primary_Pos',
   marker='o',
   linewidth=2.5,
   ax=axes[1],
   palette='Set1'
)
axes[1].set_title('Total Minutes Played by Age and Primary Position', fontsize=14, fontweight='bold', pad=15)
axes[1].set_ylabel('Total Minutes Played', fontsize=12)
axes[1].get_yaxis().set_major_formatter(plt.FuncFormatter(lambda x, loc: "{:,}".format(int(x))))
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()


# =====================================================================
# PART 2: Statistical Tests of Independence (Minutes Allocation)
# =====================================================================
print('=== Statistical Tests for Minutes Allocation by Age ===\n')

# A. Minutes by Age vs Primary Position
contingency_pos = pd.crosstab(df_clean['Age'], df_clean['Primary_Pos'], values=df_clean['Min'], aggfunc='sum').fillna(0)
chi2_pos, p_pos, dof_pos, _ = stats.chi2_contingency(contingency_pos)

print('1. Minutes Distribution by Age vs Primary Position:')
print(f'   Chi-Square test: Chi2-statistic = {chi2_pos:.4f}, p-value = {p_pos:.4e}')
if p_pos < 0.05:
   print('   -> Statistically Significant (p < 0.05). Minutes are distributed differently across ages depending on position.\n')
else:
   print('   -> Not Statistically Significant (p >= 0.05). Minutes allocation by age is independent of position.\n')

# B. Minutes by Age vs Competition
contingency_comp = pd.crosstab(df_single_comp['Age'], df_single_comp['Comp'], values=df_single_comp['Min'], aggfunc='sum').fillna(0)
chi2_comp, p_comp, dof_comp, _ = stats.chi2_contingency(contingency_comp)

print('2. Minutes Distribution by Age vs Competition (Top 5 Leagues):')
print(f'   Chi-Square test: Chi2-statistic = {chi2_comp:.4f}, p-value = {p_comp:.4e}')
if p_comp < 0.05:
   print('   -> Statistically Significant (p < 0.05). Leagues have significantly different distributions of playing time by age.\n')
else:
   print('   -> Not Statistically Significant (p >= 0.05). Leagues share a similar age playing time profile.\n')


# =====================================================================
# PART 3: Club-Level Analysis of Youth vs Experience Allocation
# =====================================================================
# exclude multi-club players
df_clean = df_clean[~df_clean.Squad.str.contains(", ")]
# Define Youth (Age <= 23) and Experience (Age >= 30)
df_clean['Is_Young'] = df_clean['Age'] <= 23
df_clean['Is_Older'] = df_clean['Age'] >= 30

# Calculate squad level statistics
squad_metrics = df_clean.groupby(['Squad', 'Comp']).agg(
   Total_Min=('Min', 'sum'),
   Weighted_Avg_Age=('Age', lambda x: np.average(x, weights=df_clean.loc[x.index, 'Min'])),
   Young_Min=('Min', lambda x: x[df_clean.loc[x.index, 'Is_Young']].sum()),
   Older_Min=('Min', lambda x: x[df_clean.loc[x.index, 'Is_Older']].sum())
).reset_index()

# Calculate proportions
squad_metrics['Pct_Young_Min'] = (squad_metrics['Young_Min'] / squad_metrics['Total_Min']) * 100
squad_metrics['Pct_Older_Min'] = (squad_metrics['Older_Min'] / squad_metrics['Total_Min']) * 100

# Global averages (weighted by minutes)
global_total_min = df_clean['Min'].sum()
global_weighted_age = np.average(df_clean['Age'], weights=df_clean['Min'])
global_young_prop = df_clean[df_clean['Is_Young']]['Min'].sum() / global_total_min
global_older_prop = df_clean[df_clean['Is_Older']]['Min'].sum() / global_total_min

print('=== European Averages ===')
print(f"Weighted Average Age on Pitch: {global_weighted_age:.2f} years old")
print(f"Percentage of Minutes to Young Players (<=23): {global_young_prop*100:.2f}%")
print(f"Percentage of Minutes to Older Players (>=30): {global_older_prop*100:.2f}%\n")


# Perform Two-Sided Binomial Proportion tests for each Squad
def run_binom_test(k, n, p_expected):
   if n == 0: return 1.0
   try:
       # Modern scipy
       return stats.binomtest(int(k), int(n), p_expected).pvalue
   except AttributeError:
       # Fallback for older scipy versions
       return stats.binom_test(int(k), int(n), p_expected)

# Test young minutes
squad_metrics['Young_P_Val'] = squad_metrics.apply(
   lambda row: run_binom_test(row['Young_Min'], row['Total_Min'], global_young_prop), axis=1
)

# Test older minutes
squad_metrics['Older_P_Val'] = squad_metrics.apply(
   lambda row: run_binom_test(row['Older_Min'], row['Total_Min'], global_older_prop), axis=1
)

# Sort out statistically significant outliers (p < 0.05)
significant_squads = squad_metrics[
   (squad_metrics['Young_P_Val'] < 0.05) | (squad_metrics['Older_P_Val'] < 0.05)
].copy()

# Helper function to format display dataframe without relying on jinja2 / .style
def display_formatted_squads(df, p_val_col):
   formatted = df.copy()
   formatted['Total_Min'] = formatted['Total_Min'].map('{:,.0f}'.format)
   formatted['Weighted_Avg_Age'] = formatted['Weighted_Avg_Age'].map('{:.2f}'.format)
   formatted['Pct_Young_Min'] = formatted['Pct_Young_Min'].map('{:.1f}%'.format)
   formatted['Pct_Older_Min'] = formatted['Pct_Older_Min'].map('{:.1f}%'.format)
   formatted[p_val_col] = formatted[p_val_col].map('{:.2e}'.format)

   cols_to_show = ['Squad', 'Comp', 'Total_Min', 'Weighted_Avg_Age', 'Pct_Young_Min', 'Pct_Older_Min', p_val_col]
   display(formatted[cols_to_show])

# ---------------------------------------------------------------------
# Display Results
# ---------------------------------------------------------------------

print('=== TOP 10 CLUBS GIVING SIGNIFICANTLY MORE MINUTES TO YOUTH (Age <= 23) ===')
youth_promoters = significant_squads[significant_squads['Pct_Young_Min'] > (global_young_prop * 100)]
youth_promoters_top = youth_promoters.sort_values(by='Pct_Young_Min', ascending=False).head(10)
display_formatted_squads(youth_promoters_top, 'Young_P_Val')

print('\n=== TOP 10 CLUBS GIVING SIGNIFICANTLY FEWER MINUTES TO YOUTH (Age <= 23) ===')
youth_avoiders = significant_squads[significant_squads['Pct_Young_Min'] < (global_young_prop * 100)]
youth_avoiders_top = youth_avoiders.sort_values(by='Pct_Young_Min', ascending=True).head(10)
display_formatted_squads(youth_avoiders_top, 'Young_P_Val')

print('\n=== TOP 10 CLUBS GIVING SIGNIFICANTLY MORE MINUTES TO EXPERIENCE (Age >= 30) ===')
exp_promoters = significant_squads[significant_squads['Pct_Older_Min'] > (global_older_prop * 100)]
exp_promoters_top = exp_promoters.sort_values(by='Pct_Older_Min', ascending=False).head(10)
display_formatted_squads(exp_promoters_top, 'Older_P_Val')

**Findings:**

- The average age is approximately 27, and is roughly normally distributed. The youngest player is Gavi (who was 17 and played 34 times for Barcelona). The oldest is Ibrahimovic, 40, who played 23 times for AC Milan.

- There are no significant differences in the the age distribution by position (means for all three are between 26.6 and 26.9 years old).

- However there is between leagues; Ligue 1 and Bundesliga players skew slightly younger (with means for both of 26.0), Serie A and Premier League players tend to be slightly older (both with an average of 26.8), however Premier League age distribution has slight signs of bi-modality. La Liga players tend to be older than average, with a mean of 27.8.

- The average minutes played by age is fairly steady across the age range, with a spike at 17 (an outlier in Gavi), a slight curve to a peak at 29, and then a steady decline. However, more minutes are played by 23 to 25 years old than any other age, with this amount dropping drastically before 22 and after 31, due to decreasing number of players active at these ages.

- The total numbers of minutes played by age does vary significantly by league and position played. Although bell-shaped regardless, the total minutes played by Forward is comparatively flatter than other positions, indicating age has a lessened effect on minutes played by age. Midfields peak around 25, and defenders peak in the late-20s, highlighting that younger players are preferred in midfield, but older players are preferred in defence.

- Similarly, Ligue 1 and Seria A gives proportionally more minutes to players in their early-20s, whereas La Liga and the Premier League give comparatively more minute to players in their late-20s and early-30s.

- Overall, 22.2% of minutes are given to players 23 and under, and 26.0% are given to players 30 or older.

- Leverkusen give the highest percentage of minutes to young players (59.7% of total minutes played). Other notable clubs in the top 10 for giving young players minutes are Arsenal (44.6%) and Barcelona (43.4%).

- Man City gave the 7th least minute to young players in the top 5 leagues (6.3%). Inter (7.0%) and Atletico Madrid (7.2%) also made the top 10.

- Watford gave the highest percentage of minutes to older players (50.2%). Real Madrid (47.8%) and Liverpool (44.2%) also made the top 10.

#### Normalising data per 90 minutes

The majority of metrics are raw counts, which can be skewed simply by playing time (e.g., a player who plays twice as many minutes may have more total passes than a player who has had limited playtime but makes far more passes per game). These counts can therefore be normalised by calculating the 'per 90 min' equivalents. Ratios, Percentages, and already-calculated-averages (e.g goals-per-shot) do not require further normalisation.

In [ ]:
columns_to_normalise = PER90_COUNT_COLUMNS

player_data_p90 = aggregated_player_data.copy()

for raw_col_name in columns_to_normalise:
    new_col_name = f"{raw_col_name}_p90"

    player_data_p90[new_col_name] = player_data_p90[raw_col_name] / player_data_p90['90s']

    player_data_p90.drop(raw_col_name, axis=1, inplace=True)

player_data_p90 = player_data_p90.copy()

player_data_p90.head()

The remaining features can now be explored.

There are 141 metrics in the dataset; far too many to explore one by one, or all at once. However, these metrics can broadly be categorised (see the data-dictionary.md for more context categories).

In [ ]:
# Column names by category
descriptive_cols = ['player_identifier', 'Player', 'Nation', 'Age', 'Pos', 'Primary_Pos', 'Num_Pos', 'Squad', 'Comp']

matches_played_cols = ['MP', 'Starts', 'Min', '90s']
finishing_cols = ['SoT%', 'G/Sh', 'ShoDist', 'Goals_p90', 'Shots_p90', 'SoT_p90']
chance_creation_cols = ['Assists_p90', 'PasAss_p90', 'SCA_p90', 'ScaPassLive_p90', 'ScaDrib_p90', 'ScaSh_p90', 'ScaFld_p90', 'ScaDef_p90', 'GCA_p90', 'GcaPassLive_p90', 'GcaDrib_p90', 'GcaSh_p90', 'GcaFld_p90', 'GcaDef_p90', 'PKwon_p90']
set_pieces_cols = ['ShoFK_p90', 'ShoPK_p90', 'PKatt_p90', 'PasDead_p90', 'PasFK_p90', 'CK_p90', 'CkIn_p90', 'CkOut_p90', 'CkStr_p90', 'TI_p90', 'ScaPassDead_p90', 'GcaPassDead_p90']
passing_cols = ['PasTotCmp%', 'PasShoCmp%', 'PasMedCmp%', 'PasLonCmp%', 'PasTotCmp_p90', 'PasTotAtt_p90', 'PasTotDist_p90', 'PasTotPrgDist_p90', 'PasShoCmp_p90', 'PasShoAtt_p90', 'PasMedCmp_p90', 'PasMedAtt_p90', 'PasLonCmp_p90', 'PasLonAtt_p90',
'Pas3rd_p90', 'PPA_p90', 'CrsPA_p90', 'PasProg_p90', 'PasAtt_p90', 'PasLive_p90', 'TB_p90', 'PasPress_p90', 'Sw_p90', 'PasCrs_p90', 'PasGround_p90', 'PasLow_p90', 'PasHigh_p90', 'PaswLeft_p90', 'PaswRight_p90', 'PaswHead_p90', 'PaswOther_p90', 'PasCmp_p90',
'PasOut_p90', 'PasInt_p90', 'PasBlocks_p90', 'RecProg_p90', 'Crs_p90']
positioning_cols = ['PasOff_p90', 'Off_p90']
defensive_plays_cols = ['TklDri%', 'Tkl_p90', 'TklWon_p90', 'TklDef3rd_p90', 'TklMid3rd_p90', 'TklAtt3rd_p90', 'TklDri_p90', 'TklDriAtt_p90', 'TklDriPast_p90', 'Blocks_p90', 'BlkSh_p90', 'BlkShSv_p90', 'BlkPass_p90', 'Int_p90', 'Tkl+Int_p90', 'Clr_p90',
'Err_p90', 'TklW_p90', 'PKcon_p90', 'OG_p90', 'Recov_p90']
pressures_cols = ['Press%', 'Press_p90', 'PresSucc_p90', 'PresDef3rd_p90', 'PresMid3rd_p90', 'PresAtt3rd_p90']
involvement_cols = ['Rec%', 'Touches_p90', 'TouDefPen_p90', 'TouDef3rd_p90', 'TouMid3rd_p90', 'TouAtt3rd_p90', 'TouAttPen_p90', 'TouLive_p90', 'RecTarg_p90', 'Rec_p90']
dribbling_cols = ['DriSucc%', 'DriSucc_p90', 'DriAtt_p90', 'DriPast_p90', 'DriMegs_p90', 'Carries_p90', 'CarTotDist_p90', 'CarPrgDist_p90', 'CarProg_p90', 'Car3rd_p90', 'CPA_p90', 'CarMis_p90', 'CarDis_p90', 'Fld_p90']
discipline_cols = ['CrdY_p90', 'CrdR_p90', '2CrdY_p90', 'Fls_p90']
aerial_ability_cols = ['AerWon%', 'AerWon_p90', 'AerLost_p90']


#### Exploration Helper Functions

Repeating the same five-part exploration by hand for every remaining category does not scale. `explore_column_group` standardises it for any list of columns: descriptive statistics (with skew and kurtosis), a histogram/KDE + boxplot grid, a correlation matrix, a `Primary_Pos`-coloured scatterplot matrix, a scaled parallel coordinates plot, and a grid of bar charts showing each column's mean by `Primary_Pos`.

`explore_outliers` highlights a specific set of players (by `player_identifier`) against the rest of the (optionally position-filtered) dataset: a parallel coordinates plot, a full scatterplot matrix, and a radar grid with one panel per highlighted player plus a dataset-average panel.

`explore_dimensionality_reduction` standardises the input columns, uses a scree plot (elbow heuristic on the eigenvalues) to choose the number of dimensions `k`, then fits both PCA and Factor Analysis with that `k`. Player scores (coloured by `Primary_Pos`) are plotted against the first two dimensions, alongside the original features' loading vectors.

#### Matches Played Exploration

The 'MP', 'Starts', 'Min', and '90s' columns all relate to the amount of football played.

- Print summary statistics (including skew and kurtosis)
- Histogram with density curve, and an integrated box plot
- Correlation heatmap of the columns
- scatterplot of MP vs Starts, coloured by Min

In [ ]:
player_data_p90[descriptive_cols + matches_played_cols].head()

In [ ]:
player_data_p90['avg_minutes_per_match'] = player_data_p90.Min / player_data_p90.MP
player_data_p90['starts_pct'] = player_data_p90.Starts / player_data_p90.MP

# Explicitly define the base columns and append to avoid duplicate appends when running multiple times
matches_played_cols = ['MP', 'Starts', 'Min', '90s', 'avg_minutes_per_match', 'starts_pct']

explore_column_group(player_data_p90, matches_played_cols)

**Findings:**
- The number of matches is played slightly left skewed, with players tending to play between 25 and 30 games (with a hard cap of 34 games in the Bundesliga season, and 38 in the others).

- Distribution of starts is more symmetrical, indicating that they are more spread between players being rotated into the starting XI.

- This is also supported by Mins being right skewed, with a smaller proportion of players playing the majority of minutes, and the rest being spread through the players. 90s is just a transformation of Min, so offers no new data.

- Average minutes per match is heavily left skewed, with most players averaging between 70 to 80 minutes, again showing a core team if often deployed but with rotations elsewhere. The percentage of games started is also left-skewed, again supporting the idea of a core team that sees rotation through the season.

- The players playing the most minutes and games are dominated by defenders, suggesting these are less likely to be subbed or rotated. This suggests that a core team of defenders are likely to be trusted, with midfield and forwards more likely to be rotated or subbed, due to the physical demands of the position, changing tactics and mid-game strategies, and the conventional wisdom that a consistent backline is able to build a strong defensive understanding.

- Many of the matches played metrics are heavily correlated, with the exception of average minutes per match & starting percentage versus matches played. This shows the split between core starting players, and players who are more likely to act as substitutes through matches.

- There is a strong link between position and all these metrics, with defensive players playing more minutes, both overall and per match, than more attacking players. This supports the previous hypothesis of attacking players being substituted to chase matches and switch up tactics, versus a preference for a consistent backline.

#### Finishing

In [ ]:
player_data_p90[descriptive_cols + finishing_cols].sample(5)


In [ ]:
explore_column_group(player_data_p90, finishing_cols)

In [ ]:
top_goalscorers = player_data_p90.nlargest(20, 'Goals_p90').player_identifier.values

explore_outliers(player_data_p90, top_goalscorers, finishing_cols)

**Findings:**
- SoT%, G/Sh, and ShoDist metrics are dominated by defenders at both the highest and lower end; this is driven by the relatively few shots these  players take, allowing per-game averages to take extreme values. This highlights that these metrics should not be the primary metrics when judging finishing ability, but rather need to be taken into context alongside raw counts and more prevalent metrics.

- Unsurprisingly, raw output columns (Goals_p90, Shots_p90, and SoT_p90) are dominated by forwards, with high profile players like Haaland, Lewandowski, Mbappe, Salah, and Benzema dominating.

- the raw output cols are highly correlated, whereas the other metrics have weaker relationships with Goals_per90. This highlights that being able to produce raw attacking output is often more important to goalscoring than pure conversion/finishing ability.

- Average shot distance is weakly negatively correlated to all other finishing metrics, unsurprisingly showing that regular longer range shooting does not translate to improved conversion or goalscoring.

- scatterplots show clear separation by position across almost all metrics, suggesting clustering is likely to be effective. Average shot distance shows that higher values tend to be midfielders, and lowest values are defenders - the latter driven by set pieces.

- with weak correlation between shot distance and other metrics, this perhaps highlights a gap in the data; more in-depth metrics (e.g. supporting analysis of long shot accuracy and conversion) would likely be of high value to separating successful long-shot takers from those who shoot from varied distances.

#### Chance Creation

In [ ]:
explore_column_group(player_data_p90, chance_creation_cols)

#### Set Pieces

In [ ]:
explore_column_group(player_data_p90, set_pieces_cols)

#### Passing

In [ ]:
explore_column_group(player_data_p90, passing_cols)

#### Positioning

In [ ]:
explore_column_group(player_data_p90, positioning_cols)


#### Defensive Actions

In [ ]:
explore_column_group(player_data_p90, defensive_plays_cols)


#### Aerial Ability

In [ ]:
explore_column_group(player_data_p90, aerial_ability_cols)


#### Pressures

In [ ]:
explore_column_group(player_data_p90, pressures_cols)


#### Match Involvement

In [ ]:
explore_column_group(player_data_p90, involvement_cols)


#### Dribbling

In [ ]:
explore_column_group(player_data_p90, dribbling_cols)


#### Discipline

In [ ]:
explore_column_group(player_data_p90, discipline_cols)


## Dimension Reduction

Due to the high correlation between metrics, dimension reduction is likely to be effective for this feature group, and offer a second experimentation path for modelling. The advantage of dimension reduction before clustering is to prevent dominance by a multiple semantically similar features, which are each given equal weight against all other individual features.

In [ ]:
mp_scaler, mp_models = explore_dimensionality_reduction(player_data_p90, matches_played_cols)

In [ ]:
reduced_data = player_data_p90[descriptive_cols].copy()

reduced_data = add_reduced_dimensions(
    df=player_data_p90,
    target_df=reduced_data,
    columns=matches_played_cols,
    fitted_scaler=mp_scaler,
    fitted_model = mp_models['pca'],
    col_names=['time_played', 'substitute_tendency'],
    signs=['Min', '-avg_minutes_per_match']
)

reduced_data.head()


Both PCA and FA can be explored. In this case, both suggest k = 2 as the optimal number of dimensions, explaining close to 100% of the variance of these features under both methods. The loadings of principal components/factors can be examined to determine the driving behaviour.

Both the first principal component and factor are driven by overall time played, and the seconds are driven by how often the player is a substitute (direction is arbitrary under dimension reduction).

With both techniques providing similar results, PCA has been chosen for improved interpretability.

In [ ]:
fin_scaler, fin_models = explore_dimensionality_reduction(player_data_p90, finishing_cols, k=2)

In [ ]:
reduced_data = add_reduced_dimensions(
    df=player_data_p90,
    target_df=reduced_data,
    columns=finishing_cols,
    fitted_scaler=fin_scaler,
    fitted_model = fin_models['pca'],
    col_names=['shooting_output', 'longshot_tendency'],
    signs=['Goals_p90', 'ShoDist']
)

reduced_data.head()